## 1. V2 데이터 로딩 및 검증

Fragrantica 향수 콘텐츠 데이터와 Atrafshan 사용자 평점 데이터를 불러오고, 데이터 구조 및 매핑 상태를 검증한다.

### 검증 항목
- 데이터셋별 행 수와 주요 컬럼 확인
- 향수 콘텐츠 사용 가능 여부 확인
- 사용자별 평점 분포 및 데이터 희소성 분석
- Fragrantica–Atrafshan 향수 ID 연결 상태 확인

본 단계에서는 모델 학습 및 평가에 필요한 데이터가 준비되었는지 점검한다.

In [1]:
from pathlib import Path
import pandas as pd

# 현재 실행 위치 확인
print("현재 실행 위치:", Path.cwd())

# 프로젝트의 V2 데이터 폴더 찾기
if (Path.cwd() / "data_v2").is_dir():
    DATA_DIR = Path.cwd() / "data_v2"
elif (Path.cwd().parent / "data_v2").is_dir():
    DATA_DIR = Path.cwd().parent / "data_v2"
else:
    raise FileNotFoundError("data_v2 폴더를 찾을 수 없습니다.")

print("V2 데이터 경로:", DATA_DIR.resolve())

# 데이터셋 로딩
perfumes = pd.read_csv(DATA_DIR / "perfumes_content.csv")
ratings = pd.read_csv(DATA_DIR / "user_ratings.csv")
linked = pd.read_csv(DATA_DIR / "user_ratings_linked.csv")
mapping = pd.read_csv(DATA_DIR / "perfume_mapping.csv")
atrafshan = pd.read_csv(DATA_DIR / "atrafshan_perfumes.csv")

# 데이터셋 확인
datasets = {
    "Fragrantica 콘텐츠": perfumes,
    "Atrafshan 평점": ratings,
    "연결된 평점": linked,
    "향수 ID 매핑": mapping,
    "Atrafshan 향수": atrafshan
}

for name, df in datasets.items():
    print(f"{name}: {len(df):,}행, {len(df.columns)}열")

print("\n평점 데이터 컬럼:")
print(ratings.columns.tolist())

print("\n평점 데이터 예시:")
display(ratings.head())

print("\n결측값 확인:")
print(ratings[["user_id", "perfume_id", "scent_rating"]].isna().sum())

현재 실행 위치: c:\Users\sumin\perfume-recommender\evaluation
V2 데이터 경로: C:\Users\sumin\perfume-recommender\data_v2
Fragrantica 콘텐츠: 140,232행, 24열
Atrafshan 평점: 5,565행, 7열
연결된 평점: 3,886행, 8열
향수 ID 매핑: 1,369행, 6열
Atrafshan 향수: 1,369행, 7열

평점 데이터 컬럼:
['user_id', 'perfume_id', 'scent_rating', 'comment_id', 'is_comment_answer', 'source_dataset', 'source_row']

평점 데이터 예시:


C:\Users\sumin\AppData\Local\Temp\ipykernel_25260\762061454.py:18: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  perfumes = pd.read_csv(DATA_DIR / "perfumes_content.csv")


,user_id,perfume_id,scent_rating,comment_id,is_comment_answer,source_dataset,source_row
0,atr:user:93040,atr:1030,8.0,128773,NO,atrafshan_data.json,1
1,atr:user:83823,atr:1030,10.0,128366,NO,atrafshan_data.json,2
2,atr:user:70512,atr:1030,1.0,123777,NO,atrafshan_data.json,17
3,atr:user:62518,atr:1030,10.0,122212,NO,atrafshan_data.json,33
4,atr:user:83877,atr:1030,8.0,120794,NO,atrafshan_data.json,38



결측값 확인:
user_id         0
perfume_id      0
scent_rating    0
dtype: int64


In [3]:
# V2 데이터 구조 및 매핑 검증
print(f"전체 향수: {len(perfumes):,}개")
print(f"연결된 평점: {len(linked):,}건")
print(f"연결된 사용자: {linked['user_id'].nunique():,}명")

content_ids = set(
    perfumes.loc[
        perfumes["content_eligible"] == True,
        "perfume_id"
    ]
)

linked_ids = set(linked["fragrantica_id"].dropna())

common_ids = content_ids & linked_ids

print(f"콘텐츠 사용 가능 향수: {len(content_ids):,}개")
print(f"연결 평점이 있는 향수: {len(linked_ids):,}개")
print(f"공통 추천 후보: {len(common_ids):,}개")

전체 향수: 140,232개
연결된 평점: 3,886건
연결된 사용자: 1,427명
콘텐츠 사용 가능 향수: 137,677개
연결 평점이 있는 향수: 757개
공통 추천 후보: 756개


In [4]:
# 사용자별 평점 분포 및 평가 가능성 확인
import numpy as np
import pandas as pd

# 사용자 ID를 문자열로 통일
ratings["user_id"] = ratings["user_id"].astype(str)
ratings["perfume_id"] = ratings["perfume_id"].astype(str)
ratings["scent_rating"] = pd.to_numeric(
    ratings["scent_rating"], errors="raise"
)

# 사용자별 평점 개수
user_counts = ratings.groupby("user_id").size()

print("전체 사용자 수:", len(user_counts))
print("전체 평점 수:", len(ratings))
print("사용자당 평점 수 중앙값:", user_counts.median())
print("사용자당 평균 평점 수:", round(user_counts.mean(), 2))

print("\n평점 개수별 사용자 분포")
summary = pd.DataFrame({
    "조건": ["1개", "2개", "3개 이상", "5개 이상", "10개 이상"],
    "사용자 수": [
        (user_counts == 1).sum(),
        (user_counts == 2).sum(),
        (user_counts >= 3).sum(),
        (user_counts >= 5).sum(),
        (user_counts >= 10).sum()
    ]
})
display(summary)

# 공통 카탈로그로 평가할 수 있는 연결 평점 확인
exact = mapping.loc[
    mapping["match_status"].eq("matched_exact")
].copy()

exact_ids = set(exact["atrafshan_id"].astype(str))
linked_eval = linked.loc[
    linked["perfume_id"].astype(str).isin(exact_ids)
    & linked["fragrantica_id"].astype(str).isin(
        perfumes["perfume_id"].astype(str)
    )
].copy()

linked_counts = linked_eval.groupby("user_id").size()

print("\n공통 카탈로그 연결 평점:", len(linked_eval))
print("연결 평점 사용자 수:", len(linked_counts))
print("연결 평점 2개 이상 사용자:", (linked_counts >= 2).sum())
print("연결 평점 3개 이상 사용자:", (linked_counts >= 3).sum())
print("연결 평점 5개 이상 사용자:", (linked_counts >= 5).sum())

print("\n사용자-향수 중복:", ratings.duplicated(
    ["user_id", "perfume_id"]
).sum())
print("평점 범위:", ratings["scent_rating"].min(),
      "~", ratings["scent_rating"].max())

전체 사용자 수: 1747
전체 평점 수: 5565
사용자당 평점 수 중앙값: 1.0
사용자당 평균 평점 수: 3.19

평점 개수별 사용자 분포


,조건,사용자 수
0,1개,947
1,2개,273
2,3개 이상,527
3,5개 이상,290
4,10개 이상,118



공통 카탈로그 연결 평점: 3886
연결 평점 사용자 수: 1427
연결 평점 2개 이상 사용자: 610
연결 평점 3개 이상 사용자: 384
연결 평점 5개 이상 사용자: 205

사용자-향수 중복: 0
평점 범위: 1.0 ~ 10.0


In [5]:
# V2 평가에 사용할 사용자-향수 평점 준비
eval_ratings = linked[
    ["user_id", "fragrantica_id", "scent_rating"]
].copy()

eval_ratings = eval_ratings.rename(
    columns={"fragrantica_id": "perfume_id"}
)

# 공통 추천 후보에 포함되는 향수만 사용
eval_ratings = eval_ratings[
    eval_ratings["perfume_id"].isin(common_ids)
].copy()

eval_ratings["scent_rating"] = pd.to_numeric(
    eval_ratings["scent_rating"], errors="coerce"
)

eval_ratings = eval_ratings.dropna(
    subset=["user_id", "perfume_id", "scent_rating"]
)

# 사용자별 평가 가능성 확인
counts = eval_ratings.groupby("user_id").size()

print("평가용 연결 평점:", len(eval_ratings))
print("평가용 사용자:", counts.size)
print("평점 2건 이상 사용자:", (counts >= 2).sum())
print("평점 3건 이상 사용자:", (counts >= 3).sum())
print("평점 5건 이상 사용자:", (counts >= 5).sum())
print(
    "사용자-향수 중복:",
    eval_ratings.duplicated(["user_id", "perfume_id"]).sum()
)

평가용 연결 평점: 3874
평가용 사용자: 1425
평점 2건 이상 사용자: 608
평점 3건 이상 사용자: 383
평점 5건 이상 사용자: 204
사용자-향수 중복: 0


## 2. Train/Test 데이터 분리

사용자별 Leave-One-Out 방식으로 학습 데이터와 테스트 데이터를 분리한다.

### 분리 조건
- Fragrantica 콘텐츠와 연결된 평점 데이터 사용
- 연결 평점이 3건 이상인 사용자 대상
- 사용자별 평점 1건을 무작위로 테스트 데이터로 분리
- 나머지 평점은 학습 데이터로 사용
- 재현성을 위해 `RANDOM_STATE = 42` 고정

### 데이터 누수 점검
- 동일한 사용자–향수 쌍의 Train/Test 중복 여부 확인
- 테스트 사용자의 학습 이력 존재 여부 확인

**분리 결과**
- 테스트 사용자: 383명
- 테스트 평점: 383건
- Train/Test 중복 사용자–향수 쌍: 0건

해당 검사는 사용자–향수 쌍의 중복을 확인하는 것으로, 모든 형태의 데이터 누수 부재를 보장하지는 않는다.

In [ ]:
# V2 사용자별 Leave-One-Out Train/Test 분리
import numpy as np
import pandas as pd

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

# 원본 데이터 보존
ratings_split = ratings.copy()
linked_split = linked.copy()

# ID 및 평점 자료형 통일
for df in [ratings_split, linked_split]:
    df["user_id"] = df["user_id"].astype(str)
    df["perfume_id"] = df["perfume_id"].astype(str)
    df["scent_rating"] = pd.to_numeric(
        df["scent_rating"], errors="raise"
    )

linked_split["fragrantica_id"] = (
    linked_split["fragrantica_id"].astype(str)
)

# V2 콘텐츠와 연결되는 향수만 평가 대상으로 사용
linked_split = linked_split[
    linked_split["fragrantica_id"].isin(common_ids)
].copy()

# 연결 평점 3건 이상 사용자
linked_counts = linked_split.groupby("user_id").size()
eligible_users = linked_counts[
    linked_counts >= 3
].index

# 사용자별 테스트 평점 1건 선택
test_indices = []

for user_id in sorted(eligible_users):
    user_rows = linked_split.index[
        linked_split["user_id"] == user_id
    ].to_numpy()

    selected = rng.choice(user_rows)
    test_indices.append(selected)

test_linked = linked_split.loc[test_indices].copy()

# 전체 Atrafshan 평점에서 테스트 쌍 제외
test_pairs = pd.MultiIndex.from_frame(
    test_linked[["user_id", "perfume_id"]]
)

rating_pairs = pd.MultiIndex.from_frame(
    ratings_split[["user_id", "perfume_id"]]
)

test_mask = rating_pairs.isin(test_pairs)
train_ratings = ratings_split.loc[~test_mask].copy()

# 콘텐츠 기반 모델의 학습용 사용자 이력
linked_pairs = pd.MultiIndex.from_frame(
    linked_split[["user_id", "perfume_id"]]
)

train_linked = linked_split.loc[
    ~linked_pairs.isin(test_pairs)
].copy()

# 데이터 누수 및 후보 검증
train_pairs = pd.MultiIndex.from_frame(
    train_ratings[["user_id", "perfume_id"]]
)

assert not train_pairs.isin(test_pairs).any()
assert len(test_linked) == len(eligible_users)
assert test_linked["fragrantica_id"].isin(common_ids).all()
assert len(train_ratings) + len(test_linked) == len(ratings_split)

print("=== V2 Train/Test 분리 결과 ===")
print("KNN 학습 평점:", len(train_ratings))
print("콘텐츠 사용자 프로필용 평점:", len(train_linked))
print("테스트 평점:", len(test_linked))
print("테스트 대상 사용자:", test_linked["user_id"].nunique())

print("\n=== 데이터 누수 검사 ===")
print("Train/Test 중복 없음")
print("테스트 향수 모두 콘텐츠 연결 가능 집합에 포함")

=== V2 Train/Test 분리 결과 ===
KNN 학습 평점: 5182
콘텐츠 사용자 프로필용 평점: 3491
테스트 평점: 383
테스트 대상 사용자: 383

=== 데이터 누수 검사 ===
Train/Test 중복 없음
테스트 향수 모두 V2 공통 후보에 포함


In [7]:
# 기존 노트북의 모델 클래스 불러오기
import ast
import json
from pathlib import Path

# 실행 위치에 관계없이 모델 노트북 찾기
candidates = [
    Path.cwd() / "perfume_recommendation.ipynb",
    Path.cwd().parent / "perfume_recommendation.ipynb",
]

notebook_path = next(
    (p for p in candidates if p.is_file()),
    None
)

if notebook_path is None:
    raise FileNotFoundError(
        "perfume_recommendation.ipynb 파일을 찾을 수 없습니다."
    )

print("모델 노트북:", notebook_path.resolve())

with open(notebook_path, encoding="utf-8") as f:
    notebook = json.load(f)

namespace = {"__name__": "__evaluation__"}

# 기존 노트북의 import와 함수/클래스 정의만 실행
for cell in notebook["cells"]:
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell.get("source", []))

    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue

    selected = [
        node for node in tree.body
        if isinstance(
            node,
            (
                ast.Import,
                ast.ImportFrom,
                ast.FunctionDef,
                ast.AsyncFunctionDef,
                ast.ClassDef,
            ),
        )
    ]

    if not selected:
        continue

    module = ast.Module(body=selected, type_ignores=[])
    ast.fix_missing_locations(module)

    exec(
        compile(module, str(notebook_path), "exec"),
        namespace,
    )

ContentBasedRecommender = namespace[
    "ContentBasedRecommender"
]

ItemKNNRecommender = namespace[
    "ItemKNNRecommender"
]

print("콘텐츠 기반 모델:", ContentBasedRecommender.__name__)
print("협업 필터링 모델:", ItemKNNRecommender.__name__)
print("두 모델 클래스 로딩 완료")

모델 노트북: C:\Users\sumin\perfume-recommender\perfume_recommendation.ipynb
콘텐츠 기반 모델: ContentBasedRecommender
협업 필터링 모델: ItemKNNRecommender
두 모델 클래스 로딩 완료


## 3. 추천 모델 학습 및 추천 가능성 검증

두 가지 추천 모델을 동일한 V2 데이터 환경에서 학습하고, 공통 추천 후보를 구성한다.

### 3.1 Content-based Filtering
- 향수 콘텐츠 특성 기반 추천
- `brand_weight = 0.2`
- 사용자 선호 평점 기준: `like_threshold = 7`

### 3.2 Item-based KNN
- 사용자 평점 이력 기반 협업 필터링
- `k = 20`
- `min_common_users = 2`
- `shrinkage = 10`
- `min_neighbors = 1`
- 인기 향수 대체 추천 비활성화 (`include_fallback = False`)

### 3.3 공통 후보 및 추천 가능성 검증
- `matched_exact` 상태의 일대일 매핑 사용
- 공통 추천 후보: 738개
- 전체 테스트 사용자: 383명
- Content-based 추천 가능: 374명
- Item-based KNN 추천 가능: 369명
- 두 모델 모두 추천 가능: 360명

추천 생성 가능 여부는 추천 정확도와 구분하여 분석한다.

In [8]:
# 두 추천 모델 학습
import time

# 필요한 변수 확인
required_vars = [
    "perfumes",
    "atrafshan",
    "train_ratings",
    "train_linked",
    "test_linked",
]

missing = [
    name for name in required_vars
    if name not in globals()
]

if missing:
    raise NameError(
        f"이전 셀에서 생성되지 않은 변수: {missing}"
    )

# 1. 콘텐츠 기반 필터링
print("=== 콘텐츠 기반 모델 학습 ===")

start = time.perf_counter()

content_model = ContentBasedRecommender(
    brand_weight=0.2
).fit(perfumes)

content_time = time.perf_counter() - start

print(f"학습 완료: {content_time:.2f}초")


# 2. 메모리 기반 협업 필터링
print("\n=== Item-based KNN 모델 학습 ===")

start = time.perf_counter()

knn_model = ItemKNNRecommender(
    k=20,
    min_common_users=2,
    shrinkage=10.0,
    min_neighbors=1
).fit(
    train_ratings,
    item_metadata=atrafshan
)

knn_time = time.perf_counter() - start

print(f"학습 완료: {knn_time:.2f}초")


# 3. 학습 결과 요약
print("\n=== 학습 결과 ===")
print("콘텐츠 기반 모델: 학습 완료")
print("협업 필터링 모델: 학습 완료")
print(f"콘텐츠 기반 학습 시간: {content_time:.2f}초")
print(f"협업 필터링 학습 시간: {knn_time:.2f}초")
print(f"보류된 테스트 평점: {len(test_linked)}개")

=== 콘텐츠 기반 모델 학습 ===
학습 완료: 1.98초

=== Item-based KNN 모델 학습 ===
학습 완료: 0.04초

=== 학습 결과 ===
콘텐츠 기반 모델: 학습 완료
협업 필터링 모델: 학습 완료
콘텐츠 기반 학습 시간: 1.98초
협업 필터링 학습 시간: 0.04초
보류된 테스트 평점: 383개


In [ ]:
# 두 추천 모델의 추천 결과 검증 (V2)
import pandas as pd

TOP_K = 10
LIKE_THRESHOLD = 7

print("=== V2 매핑 상태별 개수 ===")
print(mapping["match_status"].value_counts(dropna=False))

# 매핑에 필요한 컬럼 확인
required_columns = [
    "match_status",
    "atrafshan_id",
    "fragrantica_id",
]

missing_columns = [
    col for col in required_columns
    if col not in mapping.columns
]

if missing_columns:
    raise ValueError(
        f"매핑 데이터에 필요한 컬럼이 없습니다: {missing_columns}"
    )

# 기존 코드와 동일하게 검증된 정확 매핑만 사용
common = mapping.loc[
    mapping["match_status"].eq("matched_exact")
    & mapping["atrafshan_id"].isin(knn_model.item_ids_)
    & mapping["fragrantica_id"].isin(content_model.item_index_)
].copy()

# 완전히 동일한 매핑 중복 제거
common = common.drop_duplicates(
    subset=["atrafshan_id", "fragrantica_id"]
)

# 일대일 매핑만 사용
common = common.loc[
    ~common["atrafshan_id"].duplicated(keep=False)
    & ~common["fragrantica_id"].duplicated(keep=False)
].copy()

print("\n=== 공통 추천 후보 검증 ===")
print("공통 후보 향수:", len(common))

if common.empty:
    raise ValueError(
        "두 모델에서 공통으로 사용할 향수가 없습니다."
    )

all_status_common = mapping.loc[
    mapping["atrafshan_id"].isin(knn_model.item_ids_)
    & mapping["fragrantica_id"].isin(content_model.item_index_)
].copy()

all_status_common = all_status_common.drop_duplicates(
    subset=["atrafshan_id", "fragrantica_id"]
)

all_status_common = all_status_common.loc[
    ~all_status_common["atrafshan_id"].duplicated(keep=False)
    & ~all_status_common["fragrantica_id"].duplicated(keep=False)
].copy()

print("\n=== 매핑 조건별 후보 비교 ===")
print("정확 매핑 후보:", len(common))
print("전체 매핑 상태 기준 후보:", len(all_status_common))

print("\n전체 매핑 상태 기준 후보의 상태 분포:")
print(
    all_status_common["match_status"].value_counts(
        dropna=False
    )
)

test_ids = set(
    test_linked["fragrantica_id"]
    .dropna()
    .astype(str)
)

common_ids = set(
    common["fragrantica_id"]
    .dropna()
    .astype(str)
)

included_test_ids = test_ids & common_ids
excluded_test_ids = test_ids - common_ids

print("\n=== 테스트 향수 포함 여부 ===")
print("테스트 향수 종류:", len(test_ids))
print("공통 후보에 포함:", len(included_test_ids))
print("공통 후보에서 제외:", len(excluded_test_ids))

if excluded_test_ids:
    print("\n제외된 테스트 향수 ID 예시:")
    print(sorted(excluded_test_ids)[:10])

test_users = set(
    test_linked["user_id"].astype(str)
)

train_users = set(
    train_linked["user_id"].astype(str)
)

eligible_users = sorted(
    test_users & train_users
)

if not eligible_users:
    raise ValueError(
        "학습·테스트에 공통으로 존재하는 사용자가 없습니다."
    )

print("\n=== 평가 사용자 확인 ===")
print("테스트 사용자:", len(test_users))
print("학습 이력 보유 테스트 사용자:", len(eligible_users))

user_id = eligible_users[0]

print("\n=== 추천 결과 예시 ===")
print("평가 예시 사용자:", user_id)

cb_result = content_model.recommend_for_user(
    user_id=user_id,
    train_linked_ratings=train_linked,
    top_n=TOP_K,
    like_threshold=LIKE_THRESHOLD,
    candidate_ids=common["fragrantica_id"]
)

print("\n=== 콘텐츠 기반 추천 결과 ===")
print("추천 개수:", len(cb_result))

display(cb_result.head(TOP_K))

cf_result = knn_model.recommend(
    user_id=user_id,
    top_n=TOP_K,
    candidate_ids=common["atrafshan_id"],
    include_fallback=False
)

print("\n=== Item-based KNN 추천 결과 ===")
print("추천 개수:", len(cf_result))

display(cf_result.head(TOP_K))

print("\n=== V2 추천 결과 검증 요약 ===")
print(f"공통 추천 후보: {len(common)}개")
print(f"학습 이력 보유 테스트 사용자: {len(eligible_users)}명")
print(f"테스트 향수 중 후보 포함: {len(included_test_ids)}종")
print(f"테스트 향수 중 후보 제외: {len(excluded_test_ids)}종")
print(f"콘텐츠 기반 추천: {len(cb_result)}개")
print(f"Item-based KNN 추천: {len(cf_result)}개")

print("\n추천 결과 검증 셀 실행 완료")

=== V2 매핑 상태별 개수 ===
match_status
matched_exact      992
unmatched          342
review_required     35
Name: count, dtype: int64

=== 공통 추천 후보 검증 ===
공통 후보 향수: 738

=== 매핑 조건별 후보 비교 ===
정확 매핑 후보: 738
전체 매핑 상태 기준 후보: 738

전체 매핑 상태 기준 후보의 상태 분포:
match_status
matched_exact    738
Name: count, dtype: int64

=== 테스트 향수 포함 여부 ===
테스트 향수 종류: 251
공통 후보에 포함: 233
공통 후보에서 제외: 18

제외된 테스트 향수 ID 예시:
['fr:13965', 'fr:14994', 'fr:20566', 'fr:24685', 'fr:25289', 'fr:28428', 'fr:32706', 'fr:44360', 'fr:50747', 'fr:52205']

=== 평가 사용자 확인 ===
테스트 사용자: 383
학습 이력 보유 테스트 사용자: 383

=== 추천 결과 예시 ===
평가 예시 사용자: atr:user:10391

=== 콘텐츠 기반 추천 결과 ===
추천 개수: 10


,perfume_id,score,perfume_name,brand,gender,url
0,fr:13067,0.366256,Uomo Trussardi 2011,Trussardi,men,https://www.fragrantica.com/perfume/Trussardi/...
1,fr:43538,0.364918,Momentum Intense,Bentley,men,https://www.fragrantica.com/perfume/Bentley/Mo...
2,fr:890,0.357939,Polo,Ralph Lauren,men,https://www.fragrantica.com/perfume/Ralph-Laur...
3,fr:65581,0.353745,Armani Code Eau de Parfum,Giorgio Armani,men,https://www.fragrantica.com/perfume/Giorgio-Ar...
4,fr:15940,0.348510,Secret Mission,Baldessarini,men,https://www.fragrantica.com/perfume/Baldessari...
5,fr:75811,0.347379,H24 Eau de Parfum,Hermès,men,https://www.fragrantica.com/perfume/Hermes/H24...
6,fr:50027,0.342459,Momentum Unlimited,Bentley,men,https://www.fragrantica.com/perfume/Bentley/Mo...
7,fr:15942,0.338511,Vince Camuto for Men,Vince Camuto,men,https://www.fragrantica.com/perfume/Vince-Camu...
8,fr:33207,0.320863,Cuir Vetiver,Yves Rocher,men,https://www.fragrantica.com/perfume/Yves-Roche...
9,fr:735,0.318344,Kouros,Yves Saint Laurent,men,https://www.fragrantica.com/perfume/Yves-Saint...



=== Item-based KNN 추천 결과 ===
추천 개수: 10


,user_id,perfume_id,predicted_rating,method,n_neighbors,perfume_name,brand,url
0,atr:user:10391,atr:14993,9.0,knn,1,Sauvage Elixir,Dior,https://www.atrafshan.ir//perfume/14993-16300/...
1,atr:user:10391,atr:692,9.0,knn,1,Silver Black,AZZARO,https://www.atrafshan.ir//perfume/692-312/silv...
2,atr:user:10391,atr:2022,8.0,knn,2,Encre Noire A L'Extreme,LALIQUE,https://www.atrafshan.ir//perfume/2022-1361/en...
3,atr:user:10391,atr:1145,8.0,knn,1,Champion,DAVIDOFF,https://www.atrafshan.ir//perfume/1145-462/cha...
4,atr:user:10391,atr:1149,8.0,knn,1,Bvlgari Man,BVLGARI,https://www.atrafshan.ir//perfume/1149-379/bvl...
5,atr:user:10391,atr:1170,8.0,knn,1,Terre d'Hermes Parfum,HERMES,https://www.atrafshan.ir//perfume/1170-766/ter...
6,atr:user:10391,atr:1179,8.0,knn,1,Black Orchid,TOM FORD,https://www.atrafshan.ir//perfume/1179-710/bla...
7,atr:user:10391,atr:13057,8.0,knn,1,Oud Save The King,Atkinsons,https://www.atrafshan.ir//perfume/13057-13605/...
8,atr:user:10391,atr:137,8.0,knn,1,Happy for Men,CLINIQUE,https://www.atrafshan.ir//perfume/137-455/happ...
9,atr:user:10391,atr:149,8.0,knn,1,Jaguar for Men,JAGUAR,https://www.atrafshan.ir//perfume/149-224/jagu...



=== V2 추천 결과 검증 요약 ===
공통 추천 후보: 738개
학습 이력 보유 테스트 사용자: 383명
테스트 향수 중 후보 포함: 233종
테스트 향수 중 후보 제외: 18종
콘텐츠 기반 추천: 10개
Item-based KNN 추천: 10개

추천 결과 검증 셀 실행 완료


In [10]:
# 테스트 정답의 추천 후보 포함 여부 진단

candidate_ids = set(common["fragrantica_id"].astype(str))

test_check = test_linked.copy()
test_check["fragrantica_id"] = (
    test_check["fragrantica_id"].astype(str)
)

test_check["in_candidate"] = (
    test_check["fragrantica_id"].isin(candidate_ids)
)

print("=== 테스트 정답 후보 포함 여부 ===")
print("전체 테스트 사용자:", len(test_check))
print("후보에 포함된 테스트 건수:", test_check["in_candidate"].sum())
print("후보에서 제외된 테스트 건수:", (~test_check["in_candidate"]).sum())

print("\n후보 포함 비율:")
print(f"{test_check['in_candidate'].mean():.2%}")

print("\n제외된 테스트 향수 예시:")
display(
    test_check.loc[
        ~test_check["in_candidate"],
        ["user_id", "perfume_id", "fragrantica_id"]
    ].head(10)
)

=== 테스트 정답 후보 포함 여부 ===
전체 테스트 사용자: 383
후보에 포함된 테스트 건수: 364
후보에서 제외된 테스트 건수: 19

후보 포함 비율:
95.04%

제외된 테스트 향수 예시:


,user_id,perfume_id,fragrantica_id
3155,atr:user:23860,atr:12877,fr:57305
1239,atr:user:25022,atr:1541,fr:13965
3063,atr:user:26072,atr:6720,fr:20566
3079,atr:user:33039,atr:6573,fr:24685
3056,atr:user:42267,atr:12690,fr:63403
3023,atr:user:43152,atr:6790,fr:44360
3055,atr:user:45116,atr:12690,fr:63403
3217,atr:user:47984,atr:16015,fr:72727
3187,atr:user:50702,atr:15950,fr:72158
2995,atr:user:51459,atr:13737,fr:62491


In [11]:
# 전체 테스트 사용자 추천 가능 여부 및 KNN 신뢰도 점검
import pandas as pd

TOP_K = 10

test_users = sorted(test_linked["user_id"].astype(str).unique())

records = []
neighbor_counts = []

for user_id in test_users:
    cb = content_model.recommend_for_user(
        user_id=user_id,
        train_linked_ratings=train_linked,
        top_n=TOP_K,
        like_threshold=7,
        candidate_ids=common["fragrantica_id"]
    )

    cf = knn_model.recommend(
        user_id=user_id,
        top_n=TOP_K,
        candidate_ids=common["atrafshan_id"],
        include_fallback=False
    )

    records.append({
        "user_id": user_id,
        "content_count": len(cb),
        "knn_count": len(cf),
        "content_available": len(cb) > 0,
        "knn_available": len(cf) > 0
    })

    if not cf.empty:
        neighbor_counts.extend(
            cf["n_neighbors"].tolist()
        )

availability = pd.DataFrame(records)

print("전체 테스트 사용자:", len(availability))
print(
    "콘텐츠 기반 추천 가능:",
    availability["content_available"].sum()
)
print(
    "KNN 추천 가능:",
    availability["knn_available"].sum()
)
print(
    "두 모델 모두 추천 가능:",
    (
        availability["content_available"]
        & availability["knn_available"]
    ).sum()
)

print("\n모델별 추천 개수 통계")
display(
    availability[
        ["content_count", "knn_count"]
    ].describe()
)

print("\nKNN 이웃 수 분포")
if neighbor_counts:
    display(
        pd.Series(neighbor_counts, name="n_neighbors")
        .value_counts()
        .sort_index()
        .rename_axis("이웃 수")
        .reset_index(name="추천 건수")
    )
else:
    print("KNN 추천 결과 없음")

전체 테스트 사용자: 383
콘텐츠 기반 추천 가능: 374
KNN 추천 가능: 369
두 모델 모두 추천 가능: 360

모델별 추천 개수 통계


,content_count,knn_count
count,383.000000,383.000000
mean,9.765013,9.133159
std,1.516792,2.473000
min,0.000000,0.000000
25%,10.000000,10.000000
50%,10.000000,10.000000
75%,10.000000,10.000000
max,10.000000,10.000000



KNN 이웃 수 분포


,이웃 수,추천 건수
0,1,2792
1,2,596
2,3,83
3,4,21
4,5,2
5,6,4


In [12]:
# 테스트 정답 데이터 및 평가 조건 점검
import pandas as pd

RELEVANCE_THRESHOLD = 7.0

test = test_linked.copy()
train = train_linked.copy()

for df in [train, test]:
    df["user_id"] = df["user_id"].astype(str)
    df["perfume_id"] = df["perfume_id"].astype(str)
    df["scent_rating"] = pd.to_numeric(
        df["scent_rating"], errors="coerce"
    )

# 테스트 평점 통계
print("=== 테스트 평점 통계 ===")
print("전체 테스트 평점:", len(test))
print("테스트 사용자:", test["user_id"].nunique())
print("테스트 향수:", test["perfume_id"].nunique())

print("\n평점 분포")
display(test["scent_rating"].describe())

# 관련 아이템(좋아한 향수) 정의
relevant = test[
    test["scent_rating"] >= RELEVANCE_THRESHOLD
].copy()

print("\n=== 관련 아이템 기준 ===")
print("기준 평점:", RELEVANCE_THRESHOLD)
print("관련 테스트 평점:", len(relevant))
print("관련 아이템이 있는 사용자:",
      relevant["user_id"].nunique())

# Train/Test 중복 확인
train_pairs = set(zip(
    train["user_id"], train["perfume_id"]
))
test_pairs = set(zip(
    test["user_id"], test["perfume_id"]
))

overlap = train_pairs & test_pairs

print("\n=== 데이터 누수 점검 ===")
print("학습/테스트 중복 사용자-향수 쌍:", len(overlap))

# 테스트 사용자 학습 이력 확인
train_users = set(train["user_id"])
test_users = set(test["user_id"])

print("\n=== 평가 사용자 확인 ===")
print("학습 이력이 있는 테스트 사용자:",
      len(train_users & test_users))
print("학습 이력이 없는 테스트 사용자:",
      len(test_users - train_users))

# 테스트 평점 결측값
print("\n=== 테스트 데이터 품질 ===")
print("테스트 평점 결측값:", test["scent_rating"].isna().sum())

# 관련 아이템의 공통 추천 후보 포함 여부
common_atr_ids = set(
    common["atrafshan_id"].astype(str)
)

relevant_in_candidates = relevant[
    relevant["perfume_id"].isin(common_atr_ids)
]

print("\n=== 관련 아이템 후보 포함 여부 ===")
print("관련 테스트 평점:", len(relevant))
print(
    "공통 후보에 포함된 관련 평점:",
    len(relevant_in_candidates)
)
print(
    "공통 후보에서 제외된 관련 평점:",
    len(relevant) - len(relevant_in_candidates)
)

# 실제 평가 가능 사용자
evaluable_users = (
    set(relevant_in_candidates["user_id"])
    & set(train["user_id"])
)

print("\n=== 실제 평가 가능 사용자 ===")
print("관련 아이템 보유 사용자:", relevant["user_id"].nunique())
print("후보에 정답이 포함되고 학습 이력이 있는 사용자:",
      len(evaluable_users))

=== 테스트 평점 통계 ===
전체 테스트 평점: 383
테스트 사용자: 383
테스트 향수: 251

평점 분포


count    383.000000
mean       7.898172
std        1.743371
min        1.000000
25%        7.000000
50%        8.000000
75%        9.000000
max       10.000000
Name: scent_rating, dtype: float64


=== 관련 아이템 기준 ===
기준 평점: 7.0
관련 테스트 평점: 315
관련 아이템이 있는 사용자: 315

=== 데이터 누수 점검 ===
학습/테스트 중복 사용자-향수 쌍: 0

=== 평가 사용자 확인 ===
학습 이력이 있는 테스트 사용자: 383
학습 이력이 없는 테스트 사용자: 0

=== 테스트 데이터 품질 ===
테스트 평점 결측값: 0

=== 관련 아이템 후보 포함 여부 ===
관련 테스트 평점: 315
공통 후보에 포함된 관련 평점: 300
공통 후보에서 제외된 관련 평점: 15

=== 실제 평가 가능 사용자 ===
관련 아이템 보유 사용자: 315
후보에 정답이 포함되고 학습 이력이 있는 사용자: 300


In [13]:
# V2 성능 평가 대상 사용자 검증
import pandas as pd

RELEVANCE_THRESHOLD = 7.0

# 1. 두 모델 모두 추천 가능한 사용자
common_users = set(
    availability.loc[
        availability["content_available"]
        & availability["knn_available"],
        "user_id"
    ].astype(str)
)

# 2. 관련 테스트 항목이 있는 사용자
relevant = test_linked.loc[
    pd.to_numeric(
        test_linked["scent_rating"],
        errors="coerce"
    ) >= RELEVANCE_THRESHOLD
].copy()

relevant["user_id"] = relevant["user_id"].astype(str)
relevant["perfume_id"] = relevant["perfume_id"].astype(str)

relevant_users = set(relevant["user_id"])

# 3. 정답 향수가 공통 추천 후보에 있는 사용자
candidate_ids = set(
    common["atrafshan_id"].astype(str)
)

candidate_users = set(
    relevant.loc[
        relevant["perfume_id"].isin(candidate_ids),
        "user_id"
    ]
)

# 4. 평가 집합
evaluation_users = relevant_users & candidate_users

diagnostic_users = (
    common_users
    & relevant_users
    & candidate_users
)

print("=== 평가 대상 사용자 ===")
print("전체 테스트 사용자:", test_linked["user_id"].nunique())
print("두 모델 모두 추천 가능:", len(common_users))
print("관련 테스트 항목 보유:", len(relevant_users))
print("관련 정답이 공통 후보에 포함:", len(candidate_users))

print("\n=== 최종 평가 기준 ===")
print("주 평가 대상:", len(evaluation_users))
print("추천 가능 사용자만 따로 분석:", len(diagnostic_users))
print("관련 정답 전체 기준:", len(relevant_users))

=== 평가 대상 사용자 ===
전체 테스트 사용자: 383
두 모델 모두 추천 가능: 360
관련 테스트 항목 보유: 315
관련 정답이 공통 후보에 포함: 300

=== 최종 평가 기준 ===
주 평가 대상: 300
추천 가능 사용자만 따로 분석: 286
관련 정답 전체 기준: 315


## 4. V2 Baseline Top-10 추천 성능 평가

학습된 두 추천 모델에 동일한 후보 집합과 평가 조건을 적용하여 Top-10 추천 성능을 비교한다.

### 4.1 평가 조건
- 추천 개수: `Top-K = 10`
- 관련 아이템 기준: 테스트 평점 7점 이상
- 공통 추천 후보: 738개
- 사용자별 Leave-One-Out 테스트 정답 1건
- 평가 지표: Hit Rate@10, Precision@10, Recall@10

### 4.2 평가 대상
- 전체 테스트 사용자: 383명
- 관련 정답이 있는 사용자: 315명
- 관련 정답이 공통 후보에 포함된 사용자: 300명
- 두 모델 모두 추천 가능하고 관련 정답이 후보에 포함된 사용자: 286명

**주 평가:** 300명 기준. 추천을 생성하지 못한 경우도 실패로 반영한다.

**보조 평가:** 315명 기준. 공통 후보에 없는 관련 정답도 실패로 반영한다.

### 4.3 평가 지표
- **Hit Rate@10:** 추천 목록에 관련 정답이 하나 이상 포함된 사용자 비율
- **Precision@10:** Top-10 추천 슬롯 중 관련 정답이 차지하는 비율
- **Recall@10:** 사용자별 관련 정답 중 Top-10에 포함된 비율

사용자당 관련 테스트 정답이 1개이므로 본 평가에서는 Hit Rate@10과 Recall@10이 동일하다.

In [ ]:
# V2 Baseline: 두 추천 모델의 공통 후보 기준 Top-10 성능 평가
import pandas as pd
import time
from pathlib import Path

TOP_K = 10
RELEVANCE_THRESHOLD = 7.0

atr_to_fr = dict(zip(
    common["atrafshan_id"].astype(str),
    common["fragrantica_id"].astype(str)
))

fr_to_atr = {fr: atr for atr, fr in atr_to_fr.items()}

candidate_atr = set(atr_to_fr.keys())

test_eval = test_linked.copy()

test_eval["user_id"] = test_eval["user_id"].astype(str)
test_eval["perfume_id"] = test_eval["perfume_id"].astype(str)
test_eval["scent_rating"] = pd.to_numeric(
    test_eval["scent_rating"],
    errors="coerce"
)

# 평점 7점 이상인 관련 정답
relevant_test = test_eval.loc[
    test_eval["scent_rating"] >= RELEVANCE_THRESHOLD
].copy()

# 관련 정답이 있는 전체 사용자: 315명
all_relevant_users = sorted(
    relevant_test["user_id"].unique()
)

# 관련 정답이 공통 후보에 있는 사용자: 300명
candidate_users = set(
    relevant_test.loc[
        relevant_test["perfume_id"].isin(candidate_atr),
        "user_id"
    ]
)

# 사용자별 정답
truth_by_user = (
    relevant_test.groupby("user_id")["perfume_id"]
    .apply(set)
    .to_dict()
)

print("=== 평가 대상 ===")
print("관련 정답 전체 사용자:", len(all_relevant_users))
print("정답이 공통 후보에 있는 사용자:", len(candidate_users))

assert len(all_relevant_users) == 315
assert len(candidate_users) == 300

records = []
start = time.perf_counter()

for i, user_id in enumerate(all_relevant_users, start=1):

    truth = truth_by_user[user_id]

    # 콘텐츠 기반 추천
    cb = content_model.recommend_for_user(
        user_id=user_id,
        train_linked_ratings=train_linked,
        top_n=TOP_K,
        like_threshold=RELEVANCE_THRESHOLD,
        candidate_ids=common["fragrantica_id"]
    )

    # Item-based KNN 추천
    cf = knn_model.recommend(
        user_id=user_id,
        top_n=TOP_K,
        candidate_ids=common["atrafshan_id"],
        include_fallback=False
    )

    # 콘텐츠 기반 결과를 Atrafshan ID로 변환
    cb_ids = [
        fr_to_atr[pid]
        for pid in cb["perfume_id"].astype(str).head(TOP_K)
        if pid in fr_to_atr
    ]

    cf_ids = (
        cf["perfume_id"]
        .astype(str)
        .head(TOP_K)
        .tolist()
    )

    for model_name, recommendations in [
        ("Content-based", cb_ids),
        ("Item-based KNN", cf_ids)
    ]:
        hits = len(set(recommendations) & truth)

        records.append({
            "user_id": user_id,
            "model": model_name,
            "candidate_eligible": user_id in candidate_users,
            "hits": hits,
            "recommendation_count": len(recommendations),
            "hit_rate@10": float(hits > 0),
            "precision@10": hits / TOP_K,
            "recall@10": hits / len(truth)
        })

    if i % 25 == 0 or i == len(all_relevant_users):
        print(
            f"평가 진행: {i}/{len(all_relevant_users)}",
            flush=True
        )

results_v2 = pd.DataFrame(records)

def summarize_metrics(df, evaluation_type):
    summary = (
        df.groupby("model")
        .agg(
            users=("user_id", "nunique"),
            hit_rate_at_10=("hit_rate@10", "mean"),
            precision_at_10=("precision@10", "mean"),
            recall_at_10=("recall@10", "mean"),
            avg_recommendations=("recommendation_count", "mean")
        )
        .reset_index()
    )

    summary.insert(0, "evaluation_type", evaluation_type)
    return summary

# 주 평가: 후보에 정답이 있는 300명
summary_300 = summarize_metrics(
    results_v2.loc[results_v2["candidate_eligible"]],
    "candidate_eligible"
)

# 보조 평가: 관련 정답이 있는 전체 315명
summary_315 = summarize_metrics(
    results_v2,
    "all_relevant"
)

comparison_v2 = pd.concat(
    [summary_300, summary_315],
    ignore_index=True
)

print("\n=== V2 Baseline 성능 비교 ===")
display(comparison_v2.round(4))

print(
    "\n전체 평가 시간:",
    round(time.perf_counter() - start, 2),
    "초"
)

# 프로젝트 루트 또는 evaluation 폴더에서 실행 가능
if (Path.cwd() / "evaluation").is_dir():
    output_dir = Path.cwd() / "evaluation" / "results"
elif Path.cwd().name == "evaluation":
    output_dir = Path.cwd() / "results"
else:
    raise FileNotFoundError(
        "프로젝트 루트 또는 evaluation 폴더에서 실행해 주세요."
    )

output_dir.mkdir(parents=True, exist_ok=True)

comparison_v2.to_csv(
    output_dir / "model_comparison_v2.csv",
    index=False,
    encoding="utf-8-sig"
)

results_v2.to_csv(
    output_dir / "user_metrics_v2.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n평가 결과 저장 완료")
print("저장 경로:", output_dir.resolve())

=== 평가 대상 ===
관련 정답 전체 사용자: 315
정답이 공통 후보에 있는 사용자: 300
평가 진행: 25/315
평가 진행: 50/315
평가 진행: 75/315
평가 진행: 100/315
평가 진행: 125/315
평가 진행: 150/315
평가 진행: 175/315
평가 진행: 200/315
평가 진행: 225/315
평가 진행: 250/315
평가 진행: 275/315
평가 진행: 300/315
평가 진행: 315/315

=== V2 Baseline 성능 비교 ===


,evaluation_type,model,users,hit_rate_at_10,precision_at_10,recall_at_10,avg_recommendations
0,candidate_eligible,Content-based,300,0.0467,0.0047,0.0467,9.8000
1,candidate_eligible,Item-based KNN,300,0.0533,0.0053,0.0533,9.2533
2,all_relevant,Content-based,315,0.0444,0.0044,0.0444,9.8095
3,all_relevant,Item-based KNN,315,0.0508,0.0051,0.0508,9.2317



전체 평가 시간: 80.74 초

평가 결과 저장 완료
저장 경로: C:\Users\sumin\perfume-recommender\evaluation\results


## 5. V2 Baseline 평가 결과 및 분석

### 5.1 성능 비교

| 평가 기준 | 모델 | Hit Rate@10 | Precision@10 | Recall@10 |
|---|---|---:|---:|---:|
| 후보 내 정답 (300명) | Content-based | 0.0467 | 0.0047 | 0.0467 |
| 후보 내 정답 (300명) | Item-based KNN | 0.0533 | 0.0053 | 0.0533 |
| 관련 정답 전체 (315명) | Content-based | 0.0444 | 0.0044 | 0.0444 |
| 관련 정답 전체 (315명) | Item-based KNN | 0.0508 | 0.0051 | 0.0508 |

### 5.2 주요 결과

- 300명 기준 Content-based는 14명, Item-based KNN은 16명의 테스트 정답을 Top-10에 포함함
- Item-based KNN이 Content-based보다 Hit Rate@10 기준 약 0.67%p 높음
- 성공 사용자 차이가 2명에 불과하므로 우열을 일반화하기 어려움
- 두 모델 모두 Hit Rate@10이 약 4~5%로, 추천 정확도 개선 여지가 있음

### 5.3 평가 한계

- 사용자별 평점 이력이 적어 협업 필터링에 데이터 희소성 문제가 발생할 수 있음
- KNN 추천 결과에서 이웃 1개에 의존한 비율이 높음
- 정확 매핑과 공통 후보 제한으로 일부 테스트 정답이 추천 후보에서 제외됨
- 테스트 정답을 기준으로 파라미터를 반복 선택하면 평가 결과에 과적합될 수 있음

## 6. 하이퍼파라미터 비교 실험

V2 Baseline 모델의 추천 성능 개선을 위해 하이퍼파라미터별 성능을 비교한다.

### 6.1 실험 대상

| 모델 | 파라미터 | 실험 값 |
|---|---|---|
| Content-based | `brand_weight` | 0, 0.1, 0.2, 0.3, 0.5 |
| Content-based | `like_threshold` | 6, 7, 8, 9 |
| Item-based KNN | `shrinkage` | 0, 5, 10, 20, 50 |

### 6.2 실험 방법
- 기존 학습 데이터 내부에서 별도의 검증 데이터 분리
- 모든 설정에 동일한 검증 사용자와 추천 후보 사용
- `Top-K = 10` 고정
- 관련 정답 평점 기준 7점 이상으로 고정
- 추천을 생성하지 못한 사용자는 실패로 처리
- 설정별 Hit Rate@10, Precision@10, Recall@10 및 추천 가능률 비교

### 6.3 모델 선택 기준

검증 데이터의 Hit Rate@10을 주요 비교 지표로 사용한다.

성능이 동일하거나 유사한 경우 추천 가능률과 모델의 안정성을 함께 검토한다.

최종 선택된 파라미터는 기존 Baseline과 구분하여 기록한다.

In [15]:
# 하이퍼파라미터 탐색용 검증 데이터 분리
import numpy as np
import pandas as pd

VALIDATION_SEED = 123
rng_val = np.random.default_rng(VALIDATION_SEED)

# 기존 테스트 데이터는 사용하지 않음
validation_source = train_linked.copy()

for col in ["user_id", "perfume_id", "fragrantica_id"]:
    validation_source[col] = validation_source[col].astype(str)

validation_source["scent_rating"] = pd.to_numeric(
    validation_source["scent_rating"],
    errors="raise"
)

# 검증용 학습 이력에 최소 2개가 남도록 3건 이상 사용자 선택
val_counts = validation_source.groupby("user_id").size()
val_users = sorted(val_counts[val_counts >= 3].index)

val_indices = []

for user_id in val_users:
    user_rows = validation_source.index[
        validation_source["user_id"] == user_id
    ].to_numpy()

    val_indices.append(rng_val.choice(user_rows))

validation_linked = validation_source.loc[
    val_indices
].copy()

# 콘텐츠 모델용 검증 학습 이력
validation_pairs = pd.MultiIndex.from_frame(
    validation_linked[["user_id", "perfume_id"]]
)

source_pairs = pd.MultiIndex.from_frame(
    validation_source[["user_id", "perfume_id"]]
)

val_train_linked = validation_source.loc[
    ~source_pairs.isin(validation_pairs)
].copy()

# KNN용 검증 학습 이력
val_train_ratings = train_ratings.copy()

for col in ["user_id", "perfume_id"]:
    val_train_ratings[col] = (
        val_train_ratings[col].astype(str)
    )

rating_pairs = pd.MultiIndex.from_frame(
    val_train_ratings[["user_id", "perfume_id"]]
)

val_train_ratings = val_train_ratings.loc[
    ~rating_pairs.isin(validation_pairs)
].copy()

# 데이터 중복 검사
content_train_pairs = pd.MultiIndex.from_frame(
    val_train_linked[["user_id", "perfume_id"]]
)

knn_train_pairs = pd.MultiIndex.from_frame(
    val_train_ratings[["user_id", "perfume_id"]]
)

assert not content_train_pairs.isin(validation_pairs).any()
assert not knn_train_pairs.isin(validation_pairs).any()

print("=== 하이퍼파라미터 검증 데이터 ===")
print("검증 사용자:", len(val_users))
print("검증 정답:", len(validation_linked))
print("콘텐츠 모델 학습 이력:", len(val_train_linked))
print("KNN 학습 평점:", len(val_train_ratings))
print("기존 테스트 데이터는 사용하지 않음")


=== 하이퍼파라미터 검증 데이터 ===
검증 사용자: 271
검증 정답: 271
콘텐츠 모델 학습 이력: 3220
KNN 학습 평점: 4911
기존 테스트 데이터는 사용하지 않음


### 6.2 검증 정답 및 공통 후보 점검

기존 학습 데이터에서 사용자별 Leave-One-Out 방식으로 검증 데이터를 분리하였다.

| 검증 항목 | 결과 |
|---|---:|
| 전체 검증 사용자 | 271명 |
| 관련 정답 보유 사용자 (평점 7점 이상) | 217명 |
| 공통 추천 후보에 포함된 관련 정답 | 217건 |
| 관련 정답 후보 포함률 | 100.00% |
| Content 학습/검증 중복 | 0건 |
| KNN 학습/검증 중복 | 0건 |

**최종 검증 대상은 217명으로 설정하였다.**

모든 하이퍼파라미터 조합에 동일한 검증 사용자, 공통 후보 및 평가 지표를 적용한다. 추천 생성이 불가능한 경우도 추천 실패로 반영한다.

In [ ]:
# 검증 정답 및 공통 후보 점검

RELEVANCE_THRESHOLD = 7.0

val_check = validation_linked.copy()

val_check["user_id"] = val_check["user_id"].astype(str)
val_check["perfume_id"] = val_check["perfume_id"].astype(str)
val_check["scent_rating"] = pd.to_numeric(
    val_check["scent_rating"],
    errors="coerce"
)

# 공통 추천 후보
candidate_atr = set(
    common["atrafshan_id"].astype(str)
)

# 관련 정답: 평점 7점 이상
val_relevant = val_check.loc[
    val_check["scent_rating"] >= RELEVANCE_THRESHOLD
].copy()

# 관련 정답이 공통 후보에 포함된 사용자
val_candidate = val_relevant.loc[
    val_relevant["perfume_id"].isin(candidate_atr)
].copy()

validation_users = sorted(
    val_candidate["user_id"].unique()
)

print("=== 검증 데이터 최종 점검 ===")
print("전체 검증 사용자:", val_check["user_id"].nunique())
print("전체 검증 정답:", len(val_check))
print("관련 정답(7점 이상):", len(val_relevant))
print("공통 후보에 포함된 관련 정답:", len(val_candidate))
print("주 검증 대상 사용자:", len(validation_users))

print("\n=== 후보 포함률 ===")
if len(val_relevant) > 0:
    coverage = len(val_candidate) / len(val_relevant)
    print(f"관련 정답 후보 포함률: {coverage:.2%}")

# 검증 정답이 두 모델 학습 이력에 남아 있지 않은지 확인
val_pairs = set(zip(
    val_check["user_id"],
    val_check["perfume_id"]
))

content_pairs = set(zip(
    val_train_linked["user_id"].astype(str),
    val_train_linked["perfume_id"].astype(str)
))

knn_pairs = set(zip(
    val_train_ratings["user_id"].astype(str),
    val_train_ratings["perfume_id"].astype(str)
))

print("\n=== 검증 데이터 누수 점검 ===")
print("Content 학습/검증 중복:", len(val_pairs & content_pairs))
print("KNN 학습/검증 중복:", len(val_pairs & knn_pairs))


=== 검증 데이터 최종 점검 ===
전체 검증 사용자: 271
전체 검증 정답: 271
관련 정답(7점 이상): 217
공통 후보에 포함된 관련 정답: 217
주 검증 대상 사용자: 217

=== 후보 포함률 ===
관련 정답 후보 포함률: 100.00%

=== 검증 데이터 누수 점검 ===
Content 학습/검증 중복: 0
KNN 학습/검증 중복: 0


### 6.3 하이퍼파라미터별 추천 성능 비교

Content-based 모델의 `brand_weight`와 `like_threshold` 조합 20개, Item-based KNN의 `shrinkage` 설정 5개를 비교한다.

모든 실험에 동일한 217명의 검증 사용자와 738개의 공통 후보를 사용한다.

각 설정에 대해 검증용 학습 데이터로 모델을 새로 학습하며, 검증 정답은 학습에 사용하지 않는다.

주요 비교 지표는 Hit Rate@10이며, Precision@10, Recall@10, 추천 가능률 및 평균 추천 개수를 함께 기록한다.

In [ ]:
# 하이퍼파라미터 자동 비교 실험
import pandas as pd
import time
from pathlib import Path

TOP_K = 10
RELEVANCE_THRESHOLD = 7.0

BRAND_WEIGHTS = [0, 0.1, 0.2, 0.3, 0.5]
LIKE_THRESHOLDS = [6, 7, 8, 9]
SHRINKAGES = [0, 5, 10, 20, 50]

# 기존 공통 후보를 모든 실험에 동일하게 적용
candidate_fr = common["fragrantica_id"].astype(str).tolist()
candidate_atr = common["atrafshan_id"].astype(str).tolist()

fr_to_atr = dict(zip(candidate_fr, candidate_atr))

# 검증 정답: 평점 7점 이상, 공통 후보에 포함
val_truth = val_candidate.copy()
val_truth["user_id"] = val_truth["user_id"].astype(str)
val_truth["perfume_id"] = val_truth["perfume_id"].astype(str)

truth_by_user = (
    val_truth.groupby("user_id")["perfume_id"]
    .apply(set)
    .to_dict()
)

eval_users = sorted(truth_by_user)

assert len(eval_users) == 217
assert len(candidate_fr) == len(candidate_atr)

experiment_records = []
start_all = time.perf_counter()

def evaluate_setting(model_name, model, params):
    """동일한 검증 사용자에 대한 Top-10 성능 계산"""
    hits_total = 0
    available_total = 0
    recommendation_total = 0
    precision_total = 0
    recall_total = 0

    for user_id in eval_users:
        truth = truth_by_user[user_id]

        if model_name == "Content-based":
            rec = model.recommend_for_user(
                user_id=user_id,
                train_linked_ratings=val_train_linked,
                top_n=TOP_K,
                like_threshold=params["like_threshold"],
                candidate_ids=candidate_fr
            )

            rec_ids = [
                fr_to_atr[pid]
                for pid in rec["perfume_id"].astype(str).head(TOP_K)
                if pid in fr_to_atr
            ]

        else:
            valid_candidate_atr = list(
                dict.fromkeys(
                    pid for pid in candidate_atr
                    if pid in model.item_ids_
                )
            )

            rec = model.recommend(
                user_id=user_id,
                top_n=TOP_K,
                candidate_ids=valid_candidate_atr,
                include_fallback=False
            )

            rec_ids = (
                rec["perfume_id"]
                .astype(str)
                .head(TOP_K)
                .tolist()
            )

        hits = len(set(rec_ids) & truth)

        hits_total += int(hits > 0)
        available_total += int(len(rec_ids) > 0)
        recommendation_total += len(rec_ids)
        precision_total += hits / TOP_K
        recall_total += hits / len(truth)

    n = len(eval_users)

    return {
        "model": model_name,
        "brand_weight": params.get("brand_weight"),
        "like_threshold": params.get("like_threshold"),
        "shrinkage": params.get("shrinkage"),
        "users": n,
        "hits": hits_total,
        "hit_rate_at_10": hits_total / n,
        "precision_at_10": precision_total / n,
        "recall_at_10": recall_total / n,
        "coverage": available_total / n,
        "avg_recommendations": recommendation_total / n
    }

for brand_weight in BRAND_WEIGHTS:
    model = ContentBasedRecommender(
        brand_weight=brand_weight
    ).fit(perfumes)

    for like_threshold in LIKE_THRESHOLDS:
        params = {
            "brand_weight": brand_weight,
            "like_threshold": like_threshold
        }

        result = evaluate_setting(
            "Content-based",
            model,
            params
        )

        experiment_records.append(result)

        print(
            f"[Content] brand={brand_weight}, "
            f"like={like_threshold}, "
            f"HR@10={result['hit_rate_at_10']:.4f}",
            flush=True
        )

for shrinkage in SHRINKAGES:
    model = ItemKNNRecommender(
        k=20,
        min_common_users=2,
        shrinkage=float(shrinkage),
        min_neighbors=1
    ).fit(
        val_train_ratings,
        item_metadata=atrafshan
    )

    params = {"shrinkage": shrinkage}

    result = evaluate_setting(
        "Item-based KNN",
        model,
        params
    )

    experiment_records.append(result)

    print(
        f"[KNN] shrinkage={shrinkage}, "
        f"HR@10={result['hit_rate_at_10']:.4f}",
        flush=True
    )

hyperparameter_results = pd.DataFrame(experiment_records)

hyperparameter_results = hyperparameter_results.sort_values(
    ["model", "hit_rate_at_10", "coverage"],
    ascending=[True, False, False]
).reset_index(drop=True)

print("\n=== 전체 하이퍼파라미터 실험 결과 ===")
display(hyperparameter_results.round(4))

print("\n=== 모델별 검증 성능 1위 ===")
best_params = (
    hyperparameter_results
    .groupby("model", group_keys=False)
    .head(1)
    .copy()
)

display(best_params.round(4))

print(
    "\n전체 실험 시간:",
    round(time.perf_counter() - start_all, 2),
    "초"
)

if (Path.cwd() / "evaluation").is_dir():
    output_dir = Path.cwd() / "evaluation" / "results"
elif Path.cwd().name == "evaluation":
    output_dir = Path.cwd() / "results"
else:
    raise FileNotFoundError(
        "프로젝트 루트 또는 evaluation 폴더에서 실행하세요."
    )

output_dir.mkdir(parents=True, exist_ok=True)

hyperparameter_results.to_csv(
    output_dir / "hyperparameter_results_v2.csv",
    index=False,
    encoding="utf-8-sig"
)

best_params.to_csv(
    output_dir / "best_hyperparameters_v2.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n하이퍼파라미터 실험 결과 저장 완료")
print("저장 경로:", output_dir.resolve())

[Content] brand=0, like=6, HR@10=0.0184
[Content] brand=0, like=7, HR@10=0.0184
[Content] brand=0, like=8, HR@10=0.0184
[Content] brand=0, like=9, HR@10=0.0092
[Content] brand=0.1, like=6, HR@10=0.0323
[Content] brand=0.1, like=7, HR@10=0.0323
[Content] brand=0.1, like=8, HR@10=0.0369
[Content] brand=0.1, like=9, HR@10=0.0323
[Content] brand=0.2, like=6, HR@10=0.0553
[Content] brand=0.2, like=7, HR@10=0.0507
[Content] brand=0.2, like=8, HR@10=0.0461
[Content] brand=0.2, like=9, HR@10=0.0507
[Content] brand=0.3, like=6, HR@10=0.0553
[Content] brand=0.3, like=7, HR@10=0.0461
[Content] brand=0.3, like=8, HR@10=0.0553
[Content] brand=0.3, like=9, HR@10=0.0461
[Content] brand=0.5, like=6, HR@10=0.0599
[Content] brand=0.5, like=7, HR@10=0.0553
[Content] brand=0.5, like=8, HR@10=0.0599
[Content] brand=0.5, like=9, HR@10=0.0461
[KNN] shrinkage=0, HR@10=0.0415
[KNN] shrinkage=5, HR@10=0.0323
[KNN] shrinkage=10, HR@10=0.0369
[KNN] shrinkage=20, HR@10=0.0369
[KNN] shrinkage=50, HR@10=0.0415

=== 

,model,brand_weight,like_threshold,shrinkage,users,hits,hit_rate_at_10,precision_at_10,recall_at_10,coverage,avg_recommendations
0,Content-based,0.5,6.0,NaN,217,13,0.0599,0.0060,0.0599,0.9954,9.9539
1,Content-based,0.5,8.0,NaN,217,13,0.0599,0.0060,0.0599,0.9493,9.4931
2,Content-based,0.2,6.0,NaN,217,12,0.0553,0.0055,0.0553,0.9954,9.9539
3,Content-based,0.3,6.0,NaN,217,12,0.0553,0.0055,0.0553,0.9954,9.9539
4,Content-based,0.5,7.0,NaN,217,12,0.0553,0.0055,0.0553,0.9816,9.8157
5,Content-based,0.3,8.0,NaN,217,12,0.0553,0.0055,0.0553,0.9493,9.4931
6,Content-based,0.2,7.0,NaN,217,11,0.0507,0.0051,0.0507,0.9816,9.8157
7,Content-based,0.2,9.0,NaN,217,11,0.0507,0.0051,0.0507,0.7788,7.7880
8,Content-based,0.3,7.0,NaN,217,10,0.0461,0.0046,0.0461,0.9816,9.8157
9,Content-based,0.2,8.0,NaN,217,10,0.0461,0.0046,0.0461,0.9493,9.4931



=== 모델별 검증 성능 1위 ===


,model,brand_weight,like_threshold,shrinkage,users,hits,hit_rate_at_10,precision_at_10,recall_at_10,coverage,avg_recommendations
0,Content-based,0.5,6.0,NaN,217,13,0.0599,0.0060,0.0599,0.9954,9.9539
20,Item-based KNN,NaN,NaN,0.0,217,9,0.0415,0.0041,0.0415,0.9677,9.0507



전체 실험 시간: 682.73 초

하이퍼파라미터 실험 결과 저장 완료
저장 경로: C:\Users\sumin\perfume-recommender\evaluation\results


### 6.4 하이퍼파라미터 탐색 결과 분석

#### 실험 개요

검증 사용자 217명을 대상으로 Content-based 모델의 하이퍼파라미터 20개 조합과 Item-based KNN 모델의 하이퍼파라미터 5개 설정을 평가하였다.

평가 지표는 Hit Rate@10을 중심으로 비교하였으며, 기존 테스트 데이터는 하이퍼파라미터 선택에 사용하지 않았다.

#### Content-based 모델 결과

| 설정 | Hit Rate@10 | 추천 성공 |
|---|---:|---:|
| 기존: brand_weight=0.2, like_threshold=7 | 0.0507 | 11명 |
| 최고: brand_weight=0.5, like_threshold=6 | 0.0599 | 13명 |
| 공동 최고: brand_weight=0.5, like_threshold=8 | 0.0599 | 13명 |

브랜드 가중치를 0.5로 높였을 때 가장 높은 검증 성능을 기록하였다. 기존 설정 대비 Hit Rate@10이 약 0.92%p 증가하였다.

#### Item-based KNN 모델 결과

| shrinkage | Hit Rate@10 | 추천 성공 |
|---|---:|---:|
| 0 | 0.0415 | 9명 |
| 5 | 0.0323 | 7명 |
| 10 (기존) | 0.0369 | 8명 |
| 20 | 0.0369 | 8명 |
| 50 | 0.0415 | 9명 |

`shrinkage=0`과 `shrinkage=50`에서 공동 최고 성능을 기록하였다. 기존 설정인 `shrinkage=10` 대비 Hit Rate@10이 약 0.46%p 증가하였다.

#### 종합 분석

이번 검증 실험에서는 Content-based 모델의 최고 Hit Rate@10이 0.0599로, Item-based KNN의 최고 성능인 0.0415보다 높았다.

그러나 두 모델 모두 추천 성공 건수가 적고, 최적 설정 간 성능 차이도 크지 않았다. 따라서 검증 결과만으로 특정 모델의 우수성을 일반화하기는 어렵다.

또한 KNN은 학습 데이터에 존재하는 향수만 추천할 수 있으므로, 실제 평가 후보 집합과 정답 추천 가능 범위를 확인할 필요가 있다.

최종 모델 성능은 선택한 하이퍼파라미터를 기존 학습 데이터 전체에 적용한 뒤, 별도로 분리한 테스트 데이터에서 평가한다.

#### KNN 추천 후보 제한 및 평가 공정성

검증 데이터 분리 후 Item-based KNN의 추천 후보 집합을 확인한 결과, 기존 공통 후보 738개 중 722개가 검증 학습 데이터에 존재하였다. 나머지 16개는 검증 학습 데이터에 포함되지 않아 KNN의 추천 후보에서 제외되었다.

검증 관련 정답 217개 중 206개(94.93%)는 KNN 학습 후보에 포함되었으며, 11개는 KNN으로 추천할 수 없는 상태였다.

평가 시 두 모델 모두 동일한 217명의 검증 사용자를 대상으로 하였고, KNN이 추천할 수 없는 정답도 평가 대상에서 제외하지 않고 추천 실패로 처리하였다.

Content-based 모델은 738개의 공통 후보를 사용하였고, KNN은 학습 데이터에 존재하는 최대 722개의 후보를 사용하였다. 따라서 두 모델의 추천 가능 범위가 다르다는 점을 결과 해석에 반영하였다.

이러한 차이는 현재 KNN 구현이 학습 이력이 없는 아이템을 추천하지 못하는 한계와 관련된다.

In [ ]:
# KNN 후보 집합 확인

candidate_atr_set = set(map(str, candidate_atr))
validation_knn_items = set(
    map(str, val_train_ratings["perfume_id"])
)

valid_knn_candidates = (
    candidate_atr_set & validation_knn_items
)

missing_knn_candidates = (
    candidate_atr_set - validation_knn_items
)

print("=== KNN 후보 집합 확인 ===")
print("기존 공통 후보:", len(candidate_atr_set))
print("검증 KNN 학습 데이터에 있는 후보:", len(valid_knn_candidates))
print("검증 KNN 학습 데이터에 없는 후보:", len(missing_knn_candidates))

print("\n=== 관련 정답의 KNN 추천 가능 여부 ===")
print(
    "관련 정답 중 KNN 학습 후보에 있는 건수:",
    val_candidate["perfume_id"]
    .astype(str)
    .isin(valid_knn_candidates)
    .sum()
)
print("전체 관련 정답:", len(val_candidate))


=== KNN 후보 집합 확인 ===
기존 공통 후보: 738
검증 KNN 학습 데이터에 있는 후보: 722
검증 KNN 학습 데이터에 없는 후보: 16

=== 관련 정답의 KNN 추천 가능 여부 ===
관련 정답 중 KNN 학습 후보에 있는 건수: 206
전체 관련 정답: 217


## 7. 최적 하이퍼파라미터 적용 및 최종 테스트 평가

### 7.1 최종 평가 목적 및 설정

검증 데이터에서 선정한 최적 하이퍼파라미터가 별도로 분리한 테스트 데이터에서도 성능 개선을 보이는지 확인한다.

기존 Baseline과 최적화 모델을 동일한 학습 데이터, 테스트 사용자 및 추천 후보 집합에서 비교한다.

- Content-based Baseline: `brand_weight=0.2`, `like_threshold=7`
- Content-based 최적화: `brand_weight=0.5`, `like_threshold=6`
- Item-based KNN Baseline: `k=20`, `shrinkage=10`
- Item-based KNN 최적화: `k=20`, `shrinkage=0`
- KNN 공통 설정: `min_common_users=2`, `min_neighbors=1`
- 추천 개수: Top-10
- 관련 정답 기준: 평점 7점 이상
- 평가 지표: Hit Rate@10, Precision@10, Recall@10

최종 모델은 검증 데이터를 제외하지 않은 기존 학습 데이터 전체로 다시 학습한다. 테스트 정답은 학습 및 하이퍼파라미터 선정에 사용하지 않는다.

주 평가에서는 공통 후보에 정답이 포함된 사용자 300명을 사용하며, 보조 평가에서는 관련 정답을 가진 전체 사용자 315명을 사용한다.

In [ ]:
# 최종 테스트 데이터 및 누수 점검
import numpy as np
import pandas as pd
import time
from pathlib import Path

FINAL_TOP_K = 10
FINAL_RELEVANCE_THRESHOLD = 7.0

# ID 자료형 통일
final_common = common.copy()
final_common["atrafshan_id"] = (
    final_common["atrafshan_id"].astype(str)
)
final_common["fragrantica_id"] = (
    final_common["fragrantica_id"].astype(str)
)

final_train_ratings = train_ratings.copy()
final_train_linked = train_linked.copy()
final_test = test_linked.copy()

for df in [
    final_train_ratings,
    final_train_linked,
    final_test
]:
    df["user_id"] = df["user_id"].astype(str)
    df["perfume_id"] = df["perfume_id"].astype(str)

final_test["scent_rating"] = pd.to_numeric(
    final_test["scent_rating"],
    errors="raise"
)

# 공통 후보
final_candidate_atr = (
    final_common["atrafshan_id"].tolist()
)
final_candidate_fr = (
    final_common["fragrantica_id"].tolist()
)

assert len(final_candidate_atr) == len(set(final_candidate_atr))
assert len(final_candidate_fr) == len(set(final_candidate_fr))

final_fr_to_atr = dict(zip(
    final_candidate_fr,
    final_candidate_atr
))

# 평점 7점 이상의 관련 정답
final_relevant = final_test.loc[
    final_test["scent_rating"]
    >= FINAL_RELEVANCE_THRESHOLD
].copy()

# 공통 후보에 포함된 관련 정답
final_eligible = final_relevant.loc[
    final_relevant["perfume_id"].isin(
        set(final_candidate_atr)
    )
].copy()

# 학습/테스트 중복 검사
test_pairs = set(zip(
    final_test["user_id"],
    final_test["perfume_id"]
))

content_train_pairs = set(zip(
    final_train_linked["user_id"],
    final_train_linked["perfume_id"]
))

knn_train_pairs = set(zip(
    final_train_ratings["user_id"],
    final_train_ratings["perfume_id"]
))

content_overlap = len(
    test_pairs & content_train_pairs
)
knn_overlap = len(
    test_pairs & knn_train_pairs
)

print("=== 최종 테스트 데이터 점검 ===")
print("전체 테스트 사용자:", final_test["user_id"].nunique())
print("관련 정답 사용자:", final_relevant["user_id"].nunique())
print("주 평가 사용자:", final_eligible["user_id"].nunique())
print("공통 추천 후보:", len(final_common))

print("\n=== 데이터 누수 점검 ===")
print("Content 학습/테스트 중복:", content_overlap)
print("KNN 학습/테스트 중복:", knn_overlap)

assert content_overlap == 0
assert knn_overlap == 0

assert final_eligible["user_id"].nunique() == 300
assert final_relevant["user_id"].nunique() == 315

print("\n최종 평가 데이터 준비 완료")


=== 최종 테스트 데이터 점검 ===
전체 테스트 사용자: 383
관련 정답 사용자: 315
주 평가 사용자: 300
공통 추천 후보: 738

=== 데이터 누수 점검 ===
Content 학습/테스트 중복: 0
KNN 학습/테스트 중복: 0

최종 평가 데이터 준비 완료


In [ ]:
# 최종 평가용 모델 학습

final_models = {}

print("=== Content-based 모델 학습 ===")

final_models["Content-based Baseline"] = (
    ContentBasedRecommender(
        brand_weight=0.2
    ).fit(perfumes)
)

final_models["Content-based Tuned"] = (
    ContentBasedRecommender(
        brand_weight=0.5
    ).fit(perfumes)
)

print("Content-based 2개 모델 학습 완료")

print("\n=== Item-based KNN 모델 학습 ===")

final_models["Item-based KNN Baseline"] = (
    ItemKNNRecommender(
        k=20,
        min_common_users=2,
        shrinkage=10.0,
        min_neighbors=1
    ).fit(
        final_train_ratings,
        item_metadata=atrafshan
    )
)

final_models["Item-based KNN Tuned"] = (
    ItemKNNRecommender(
        k=20,
        min_common_users=2,
        shrinkage=0.0,
        min_neighbors=1
    ).fit(
        final_train_ratings,
        item_metadata=atrafshan
    )
)

print("KNN 2개 모델 학습 완료")

# KNN 추천 가능 후보 확인
for name in [
    "Item-based KNN Baseline",
    "Item-based KNN Tuned"
]:
    model = final_models[name]
    model_items = set(map(str, model.item_ids_))

    valid_candidates = [
        pid for pid in final_candidate_atr
        if pid in model_items
    ]

    print(
        name,
        "| 학습 후보:",
        len(valid_candidates),
        "/",
        len(final_candidate_atr)
    )


=== Content-based 모델 학습 ===
Content-based 2개 모델 학습 완료

=== Item-based KNN 모델 학습 ===
KNN 2개 모델 학습 완료
Item-based KNN Baseline | 학습 후보: 738 / 738
Item-based KNN Tuned | 학습 후보: 738 / 738


In [ ]:
# 최종 테스트 평가 함수

FINAL_CONFIGS = [
    {
        "name": "Content-based Baseline",
        "family": "Content-based",
        "like_threshold": 7,
        "model": final_models["Content-based Baseline"]
    },
    {
        "name": "Content-based Tuned",
        "family": "Content-based",
        "like_threshold": 6,
        "model": final_models["Content-based Tuned"]
    },
    {
        "name": "Item-based KNN Baseline",
        "family": "Item-based KNN",
        "like_threshold": None,
        "model": final_models["Item-based KNN Baseline"]
    },
    {
        "name": "Item-based KNN Tuned",
        "family": "Item-based KNN",
        "like_threshold": None,
        "model": final_models["Item-based KNN Tuned"]
    }
]


def evaluate_final_model(config, evaluation_data, evaluation_type):
    model = config["model"]
    model_name = config["name"]

    truth_by_user = (
        evaluation_data
        .groupby("user_id")["perfume_id"]
        .apply(set)
        .to_dict()
    )

    # KNN은 학습된 아이템만 후보로 사용
    if config["family"] == "Item-based KNN":
        model_items = set(map(str, model.item_ids_))

        knn_candidates = list(dict.fromkeys(
            pid for pid in final_candidate_atr
            if pid in model_items
        ))

        candidate_count = len(knn_candidates)

    else:
        knn_candidates = None
        candidate_count = len(final_candidate_fr)

    user_records = []

    for user_id in sorted(truth_by_user):
        truth = truth_by_user[user_id]

        if config["family"] == "Content-based":
            rec = model.recommend_for_user(
                user_id=user_id,
                train_linked_ratings=final_train_linked,
                top_n=FINAL_TOP_K,
                like_threshold=config["like_threshold"],
                candidate_ids=final_candidate_fr
            )

            # Fragrantica ID -> Atrafshan ID
            rec_ids = [
                final_fr_to_atr[pid]
                for pid in rec["perfume_id"].astype(str).head(FINAL_TOP_K)
                if pid in final_fr_to_atr
            ]

        else:
            if len(knn_candidates) == 0:
                rec_ids = []
            else:
                rec = model.recommend(
                    user_id=user_id,
                    top_n=FINAL_TOP_K,
                    candidate_ids=knn_candidates,
                    include_fallback=False
                )

                rec_ids = (
                    rec["perfume_id"]
                    .astype(str)
                    .head(FINAL_TOP_K)
                    .tolist()
                )

        # 중복 추천 ID 제거
        rec_ids = list(dict.fromkeys(rec_ids))

        hit_count = len(set(rec_ids) & truth)

        user_records.append({
            "evaluation_type": evaluation_type,
            "model": model_name,
            "user_id": user_id,
            "relevant_count": len(truth),
            "recommended_count": len(rec_ids),
            "hits": hit_count,
            "hit_rate_at_10": int(hit_count > 0),
            "precision_at_10": hit_count / FINAL_TOP_K,
            "recall_at_10": hit_count / len(truth),
            "has_recommendation": int(len(rec_ids) > 0),
        })

    user_df = pd.DataFrame(user_records)

    summary = {
        "evaluation_type": evaluation_type,
        "model": model_name,
        "users": len(user_df),
        "hits": int(user_df["hits"].sum()),
        "hit_rate_at_10": user_df["hit_rate_at_10"].mean(),
        "precision_at_10": user_df["precision_at_10"].mean(),
        "recall_at_10": user_df["recall_at_10"].mean(),
        "coverage": user_df["has_recommendation"].mean(),
        "avg_recommendations": user_df["recommended_count"].mean(),
        "candidate_count": candidate_count,
    }

    return summary, user_df


print("최종 평가 함수 준비 완료")


최종 평가 함수 준비 완료


In [ ]:
# 최종 테스트 평가 실행 및 결과 저장

final_summaries = []
final_user_results = []

start_time = time.perf_counter()

evaluation_sets = [
    ("candidate_eligible", final_eligible),
    ("all_relevant", final_relevant),
]

for evaluation_type, evaluation_data in evaluation_sets:
    print(f"\n=== {evaluation_type} 평가 시작 ===")

    for config in FINAL_CONFIGS:
        summary, user_df = evaluate_final_model(
            config=config,
            evaluation_data=evaluation_data,
            evaluation_type=evaluation_type
        )

        final_summaries.append(summary)
        final_user_results.append(user_df)

        print(
            f"{config['name']} | "
            f"users={summary['users']} | "
            f"HR@10={summary['hit_rate_at_10']:.4f} | "
            f"coverage={summary['coverage']:.4f}",
            flush=True
        )

final_comparison = pd.DataFrame(final_summaries)

final_user_metrics = pd.concat(
    final_user_results,
    ignore_index=True
)

print("\n=== 최종 테스트 성능 비교 ===")
display(final_comparison.round(4))

# 저장 경로
if (Path.cwd() / "evaluation").is_dir():
    final_output_dir = Path.cwd() / "evaluation" / "results"
elif Path.cwd().name == "evaluation":
    final_output_dir = Path.cwd() / "results"
else:
    raise FileNotFoundError(
        "프로젝트 루트 또는 evaluation 폴더에서 실행하세요."
    )

final_output_dir.mkdir(
    parents=True,
    exist_ok=True
)

final_comparison.to_csv(
    final_output_dir / "final_model_comparison_v2.csv",
    index=False,
    encoding="utf-8-sig"
)

final_user_metrics.to_csv(
    final_output_dir / "final_user_metrics_v2.csv",
    index=False,
    encoding="utf-8-sig"
)

elapsed = time.perf_counter() - start_time

print("\n최종 평가 완료")
print(f"실행 시간: {elapsed:.2f}초")
print("저장 위치:", final_output_dir.resolve())


=== candidate_eligible 평가 시작 ===
Content-based Baseline | users=300 | HR@10=0.0467 | coverage=0.9800
Content-based Tuned | users=300 | HR@10=0.0633 | coverage=0.9900
Item-based KNN Baseline | users=300 | HR@10=0.0533 | coverage=0.9733
Item-based KNN Tuned | users=300 | HR@10=0.0567 | coverage=0.9733

=== all_relevant 평가 시작 ===
Content-based Baseline | users=315 | HR@10=0.0444 | coverage=0.9810
Content-based Tuned | users=315 | HR@10=0.0603 | coverage=0.9905
Item-based KNN Baseline | users=315 | HR@10=0.0508 | coverage=0.9714
Item-based KNN Tuned | users=315 | HR@10=0.0540 | coverage=0.9714

=== 최종 테스트 성능 비교 ===


,evaluation_type,model,users,hits,hit_rate_at_10,precision_at_10,recall_at_10,coverage,avg_recommendations,candidate_count
0,candidate_eligible,Content-based Baseline,300,14,0.0467,0.0047,0.0467,0.9800,9.8000,738
1,candidate_eligible,Content-based Tuned,300,19,0.0633,0.0063,0.0633,0.9900,9.9000,738
2,candidate_eligible,Item-based KNN Baseline,300,16,0.0533,0.0053,0.0533,0.9733,9.2533,738
3,candidate_eligible,Item-based KNN Tuned,300,17,0.0567,0.0057,0.0567,0.9733,9.2533,738
4,all_relevant,Content-based Baseline,315,14,0.0444,0.0044,0.0444,0.9810,9.8095,738
5,all_relevant,Content-based Tuned,315,19,0.0603,0.0060,0.0603,0.9905,9.9048,738
6,all_relevant,Item-based KNN Baseline,315,16,0.0508,0.0051,0.0508,0.9714,9.2317,738
7,all_relevant,Item-based KNN Tuned,315,17,0.0540,0.0054,0.0540,0.9714,9.2317,738



최종 평가 완료
실행 시간: 310.47초
저장 위치: C:\Users\sumin\perfume-recommender\evaluation\results


In [ ]:
# Baseline 대비 최적화 모델 개선 폭

main_results = final_comparison.loc[
    final_comparison["evaluation_type"] == "candidate_eligible"
].copy()

model_pairs = [
    (
        "Content-based",
        "Content-based Baseline",
        "Content-based Tuned"
    ),
    (
        "Item-based KNN",
        "Item-based KNN Baseline",
        "Item-based KNN Tuned"
    )
]

improvement_records = []

for family, baseline_name, tuned_name in model_pairs:
    baseline = main_results.loc[
        main_results["model"] == baseline_name
    ].iloc[0]

    tuned = main_results.loc[
        main_results["model"] == tuned_name
    ].iloc[0]

    improvement_records.append({
        "model_family": family,
        "baseline_hr_at_10": baseline["hit_rate_at_10"],
        "tuned_hr_at_10": tuned["hit_rate_at_10"],
        "improvement_pp": (
            tuned["hit_rate_at_10"]
            - baseline["hit_rate_at_10"]
        ) * 100,
        "baseline_hits": baseline["hits"],
        "tuned_hits": tuned["hits"],
        "additional_hits": tuned["hits"] - baseline["hits"]
    })

final_improvement = pd.DataFrame(
    improvement_records
)

print("=== Baseline 대비 최적화 성능 변화 ===")
display(final_improvement.round(4))

final_improvement.to_csv(
    final_output_dir / "final_improvement_v2.csv",
    index=False,
    encoding="utf-8-sig"
)

print("개선 폭 결과 저장 완료")


=== Baseline 대비 최적화 성능 변화 ===


,model_family,baseline_hr_at_10,tuned_hr_at_10,improvement_pp,baseline_hits,tuned_hits,additional_hits
0,Content-based,0.0467,0.0633,1.6667,14,19,5
1,Item-based KNN,0.0533,0.0567,0.3333,16,17,1


개선 폭 결과 저장 완료


### 최종 테스트 평가 결과 및 분석

#### 평가 목적

검증 데이터에서 선정한 하이퍼파라미터를 적용한 추천 모델이 기존 Baseline 대비 최종 테스트 데이터에서도 성능 개선을 보이는지 확인하였다.

#### 평가 조건

- 주 평가: 관련 정답이 공통 추천 후보에 포함된 사용자 300명
- 보조 평가: 관련 정답을 보유한 전체 사용자 315명
- 관련 정답 기준: 평점 7점 이상
- 추천 개수: Top-10
- 평가 지표: Hit Rate@10, Precision@10, Recall@10
- 최종 모델 학습 데이터: 기존 Train/Test 분리에서 확보한 전체 학습 데이터

#### 최종 성능 비교 (300명 기준)

| 모델 | Hit Rate@10 | Precision@10 | Recall@10 | 추천 성공 |
|---|---:|---:|---:|---:|
| Content-based Baseline | 0.0467 | 0.0047 | 0.0467 | 14명 |
| **Content-based Tuned** | **0.0633** | **0.0063** | **0.0633** | **19명** |
| Item-based KNN Baseline | 0.0533 | 0.0053 | 0.0533 | 16명 |
| Item-based KNN Tuned | 0.0567 | 0.0057 | 0.0567 | 17명 |

#### Baseline 대비 개선 효과

Content-based 모델은 `brand_weight=0.2`, `like_threshold=7`에서 `brand_weight=0.5`, `like_threshold=6`으로 변경한 결과, Hit Rate@10이 0.0467에서 0.0633으로 증가하였다.

이는 약 1.67%p의 개선이며, 추천 성공 사용자가 14명에서 19명으로 증가한 결과이다.

Item-based KNN은 `shrinkage=10`에서 `shrinkage=0`으로 변경한 결과, Hit Rate@10이 0.0533에서 0.0567로 증가하였다.

이는 약 0.33%p의 개선이며, 추천 성공 사용자가 16명에서 17명으로 증가한 결과이다.

#### 전체 관련 정답 기준 평가 (315명)

| 모델 | Hit Rate@10 | 추천 성공 |
|---|---:|---:|
| Content-based Baseline | 0.0444 | 14명 |
| **Content-based Tuned** | **0.0603** | **19명** |
| Item-based KNN Baseline | 0.0508 | 16명 |
| Item-based KNN Tuned | 0.0540 | 17명 |

전체 관련 정답 기준에서도 Content-based Tuned 모델이 가장 높은 성능을 기록하였다.

#### 결과 해석

1. 두 추천 모델 모두 검증 데이터에서 선정한 하이퍼파라미터를 적용한 후 테스트 성능이 개선되었다.
2. 최적화 이전에는 Item-based KNN의 성능이 더 높았으나, 최적화 이후에는 Content-based 모델이 가장 높은 Hit Rate@10을 기록하였다.
3. Content-based 모델의 브랜드 가중치 증가가 추천 성능 개선에 기여했을 가능성이 있다.
4. KNN은 사용자 평점 데이터가 희소하고 학습 이력이 없는 아이템을 추천할 수 없다는 제약이 있다.
5. 전체적인 추천 성공률은 여전히 낮으며, 데이터 확장과 추천 알고리즘 개선이 필요하다.

#### 실험의 한계

- 사용자당 하나의 정답을 보류하는 Leave-One-Out 평가 방식
- 제한된 공통 추천 후보 집합
- 사용자-아이템 상호작용 데이터의 희소성
- KNN의 학습 이력이 없는 아이템에 대한 추천 제한
- 단일 데이터 분할에서 얻은 결과이므로 성능 차이의 통계적 유의성 미확인

#### 결론

최종 테스트 평가에서 Content-based Tuned 모델이 Hit Rate@10 **0.0633**으로 가장 높은 성능을 기록하였다.

본 실험에서는 Content-based 모델의 하이퍼파라미터 최적화가 Item-based KNN보다 큰 성능 개선을 보였으며, 현재 데이터와 평가 조건에서 Content-based Tuned 모델이 가장 우수한 결과를 나타냈다.